# 04 - Error analysis, calibration, actions, and latency

This notebook reviews the 100,023-row mock test results, missed fraud, false alerts, calibration, expected-cost actions, per-fraud-kind outcomes, and the Twi warning gate.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

report = json.loads((PROJECT_ROOT / "reports" / "metrics" / "latest.json").read_text(encoding="utf-8"))
decisions = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "test_decisions.csv")
source = pd.read_csv(PROJECT_ROOT / "data" / "mock_full" / "transactions.csv")
print("Source mock transactions:", len(source))
print("Source fraud rows:", int(source["is_fraud"].sum()))
print("Report transaction rows:", report["rows"]["all_transactions"])
assert len(source) == report["rows"]["all_transactions"] == 100023
print(report["warning"])

Source mock transactions: 100023
Source fraud rows: 22005
Report transaction rows: 100023
Mock-data metrics are rehearsal results, not evidence of real-world performance.


## Model comparison at validation-selected thresholds

In [2]:
metric_names = ["pr_auc", "roc_auc", "precision", "recall", "false_positive_rate", "amount_weighted_recall", "amount_stopped", "threshold"]
display(pd.DataFrame({name: {key: values[key] for key in metric_names}
                      for name, values in report["models"].items()}).T)

,pr_auc,roc_auc,precision,recall,false_positive_rate,amount_weighted_recall,amount_stopped,threshold
rules,0.490166,0.749330,1.0,0.030969,0.0,0.071418,634176.05,0.571429
logistic,0.987218,0.995508,1.0,0.015709,0.0,0.010541,93605.90,1.000000
gradient_boosted,0.992576,0.997646,0.0,0.000000,0.0,0.000000,0.00,0.999996
ensemble_calibrated,0.994518,0.997841,1.0,0.512343,0.0,0.632918,5620202.35,1.000000


## Calibrated ensemble recall by fraud kind

In [3]:
display(pd.DataFrame(report["models"]["ensemble_calibrated"]["per_fraud_kind"]).T)

,count,recall,amount_weighted_recall
agent_fraud,333.0,0.642643,0.706960
fake_merchant,644.0,0.338509,0.342171
fake_promo,168.0,0.166667,0.167315
limit_splitting,1262.0,0.634707,0.635232
mule_chain,904.0,0.657080,0.683306
pin_phishing,346.0,0.355491,0.522482
relative_impersonation,163.0,0.233129,0.298769
reversal_scam,175.0,0.400000,0.463073
rtp_abuse,173.0,0.502890,0.567307
sim_swap,288.0,0.381944,0.597306


## Missed fraud and false alerts

In [4]:
missed = decisions[(decisions["is_fraud"] == 1) & (decisions["action"] == "allow")].copy()
false_alerts = decisions[(decisions["is_fraud"] == 0) & (decisions["action"] != "allow")].copy()
print("Missed fraud:", len(missed))
print("False alerts:", len(false_alerts))
display(missed.sort_values("expected_loss", ascending=False).head(20))
display(false_alerts.sort_values("expected_loss", ascending=False).head(20))

Missed fraud: 40
False alerts: 540


,probability,expected_loss,tier,action,is_fraud,fraud_scenario,tx_id
12669,0.040161,4.929317,low,allow,1,fake_merchant,TX0092713
18284,0.106383,4.911702,low,allow,1,fake_merchant,TX0098328
17002,0.040161,4.492771,low,allow,1,fake_merchant,TX0097046
16936,0.013746,4.123711,low,allow,1,fake_promo,TX0096980
19708,0.040161,3.984337,low,allow,1,fake_merchant,TX0099752
12322,0.040161,3.736546,low,allow,1,fake_merchant,TX0092366
17825,0.063158,2.800421,low,allow,1,fake_merchant,TX0097869
14032,0.018433,2.764977,low,allow,1,relative_impersonation,TX0094076
13323,0.018433,2.764977,low,allow,1,fake_promo,TX0093367
15788,0.018433,2.764977,low,allow,1,fake_promo,TX0095832


,probability,expected_loss,tier,action,is_fraud,fraud_scenario,tx_id
2185,0.961538,4647.298077,critical,freeze,0,legitimate,TX0082228
6593,0.910256,4535.088590,critical,freeze,0,legitimate,TX0086637
18645,0.791667,3921.204167,critical,freeze,0,legitimate,TX0098689
14752,0.791667,2473.942500,critical,freeze,0,legitimate,TX0094796
2642,0.666667,2013.746667,critical,freeze,0,legitimate,TX0082686
10529,0.996071,1894.257583,critical,freeze,0,legitimate,TX0090573
10185,0.791667,1660.006250,critical,freeze,0,legitimate,TX0090229
518,0.791667,1460.973333,critical,freeze,0,legitimate,TX0080542
5628,0.666667,1430.046667,critical,freeze,0,legitimate,TX0085672
11040,0.791667,1191.117917,critical,freeze,0,legitimate,TX0091084


## Probability calibration and honest/ fraud action counts

In [5]:
bins = np.linspace(0, 1, 11)
calibration = decisions.copy()
calibration["probability_bin"] = pd.cut(calibration["probability"], bins=bins, include_lowest=True)
display(calibration.groupby("probability_bin", observed=False).agg(
    rows=("is_fraud", "size"),
    mean_probability=("probability", "mean"),
    observed_fraud_rate=("is_fraud", "mean"),
))
display(decisions.groupby("action").agg(
    transactions=("tx_id", "size"),
    fraud_transactions=("is_fraud", "sum"),
    mean_probability=("probability", "mean"),
    mean_expected_loss=("expected_loss", "mean"),
))

,rows,mean_probability,observed_fraud_rate
probability_bin,,,
"(-0.001, 0.1]",15378,0.001796,0.003772
"(0.1, 0.2]",105,0.136727,0.238095
"(0.2, 0.3]",133,0.216622,0.481203
"(0.3, 0.4]",3,0.328431,1.000000
"(0.4, 0.5]",1,0.433075,1.000000
"(0.5, 0.6]",9,0.555556,0.666667
"(0.6, 0.7]",55,0.663392,0.909091
"(0.7, 0.8]",295,0.789000,0.928814
"(0.8, 0.9]",44,0.836024,0.909091


,transactions,fraud_transactions,mean_probability,mean_expected_loss
action,,,,
allow,15023,40,0.001125,0.080915
freeze,3492,3416,0.974170,2492.757503
hold,938,820,0.817594,127.101662
warn,526,180,0.214601,20.437829


## Saved expected-cost action summary

In [6]:
display(pd.DataFrame(report["expected_cost_actions"]).T)

,transaction_count,fraud_count
allow,15023,40
freeze,3492,3416
hold,938,820
warn,526,180


## Latency and Twi warning review gate

The p95 value below is warmed in-process model scoring. It excludes feature replay, API/network transport, database state, and response serialization.

In [7]:
print("In-process latency:", report["latency_benchmark"])
import yaml
with open(PROJECT_ROOT / "configs" / "warnings.yaml", "r", encoding="utf-8") as file:
    warning_settings = yaml.safe_load(file)
display(pd.DataFrame(warning_settings["languages"]).T)

from momo_fraud.explain import warning_text
print("English template:", warning_text("en"))
try:
    print("Twi warning:", warning_text("tw"))
except ValueError as error:
    print("Twi warning blocked:", error)

In-process latency: {'scope': 'warmed in-process scoring; excludes feature construction and network', 'transactions': 200, 'p50_ms': 31.28440002910793, 'p95_ms': 51.29821999871637, 'p99_ms': 130.20868102903475}


,status,reviewer,reviewed_on,text
en,template,,,This transaction looks unusual. Confirm the re...
tw,pending_native_review,,,


English template: This transaction looks unusual. Confirm the recipient and never share your PIN or one-time code.
Twi warning blocked: No warning text is configured for language: tw
